In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import transforms
from tqdm import tqdm
import os
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import timm
from sklearn.model_selection import train_test_split

In [ ]:
import timm
models = [m for m in timm.list_models() if 'convnextv2' in m]
models

In [ ]:
#UTK
path = '/kaggle/input/utkface-new/UTKFace'
files = os.listdir(path)
age_list = []
file_names = []
for file in files:
    age = file.split('_')[0]
    age_list.append(int(age))
    file_names.append(f'/kaggle/input/utkface-new/UTKFace/{file}')
data = {"file_name": file_names, "age": age_list}
df = pd.DataFrame(data)
print(df)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
class AgeDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'file_name']
        label = int(self.df.loc[idx, 'age'])

        image = Image.open(img_path).convert('RGB')

        if image:
            image = self.transform(image)
            
        return image, label

In [ ]:
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)
train_data, val_data = train_test_split(train_data, test_size=0.2, random_state=42)

train_dataset = AgeDataset(train_data, transform)
test_dataset = AgeDataset(test_data, transform_test)
val_dataset = AgeDataset(val_data, transform_test)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader= DataLoader(test_dataset, batch_size=64)
val_loader = DataLoader(val_dataset, batch_size=64)

In [ ]:
def show_images(data_loader):
    images, labels = next(iter(data_loader))
    fig, axes = plt.subplots(1, 5, figsize=(15, 4))
    for i in range(5):
        img = images[i].permute(1, 2, 0)
        axes[i].imshow(img)
        axes[i].set_title(f"Клас: {labels[i].item()}")
        axes[i].axis('off')
    plt.show()

show_images(train_loader)

In [ ]:
def train(model, train_loader, val_loader, criterion, optimizer,epochs=30, device='cpu'):
    best_model_path = "/kaggle/working/best_model.pth"
    epoch_without_improvement = 0
    best_val_loss = 1000
    train_loss = []
    val_loss = []
    

    for epoch in range(1, epochs+1):
        total_samples = 0
        model.train()
        total_train_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device).float().unsqueeze(1)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_train_loss +=loss.item()
            
        avg_train_loss = total_train_loss/len(train_loader)
        
        train_loss.append(avg_train_loss)

        model.eval()
        total_val_loss =0.0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device).float().unsqueeze(1)
                outputs = model(images)
                loss = criterion(outputs, labels)
                total_val_loss += loss.item() * images.size(0) 
                total_samples  += labels.size(0)
        
        avg_val_loss = total_val_loss / total_samples 
        scheduler.step(avg_val_loss)
        val_loss.append(avg_val_loss)
        print(
                f"[epoch {epoch}/{epochs}] "
                f"Train loss {avg_train_loss:.4f} | "
                f"Val loss {avg_val_loss:.4f} |"
        )


        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), best_model_path)
            print(f'Best model with updated with loss: {best_val_loss:.4f}')
            epoch_without_improvement = 0
        else:
            epoch_without_improvement+=1

        if epoch_without_improvement > 3:
            print(f"Early stopping after {epoch} epochs without improvement.")
            return train_loss, val_loss
            
    return train_loss, val_loss
        
            
            

In [ ]:
import torch
import torch.nn as nn
import timm


# Complete fresh start
if 'model' in globals():
    del model
torch.cuda.empty_cache()

class MyModel(nn.Module):
    def __init__(self, num_classes=1):
        super(MyModel, self).__init__()
        self.backbone = timm.create_model('coatnet_0_rw_224', pretrained=True, num_classes=0)
        num_features = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.6),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        output = self.classifier(features)
        return output

In [ ]:
model = MyModel(num_classes = 1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

for param in model.backbone.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True


criterion = nn.L1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, min_lr = 1e-7, verbose=True,
)
train_losses, val_losses = train(model, train_loader, val_loader, criterion, optimizer, epochs=50, device=device) 

In [ ]:
for i, param in enumerate(model.backbone.parameters()):
    print(i,param.requires_grad)

In [ ]:
for param in model.backbone.parameters():
    param.requires_grad = True
for param in model.classifier.parameters():
    param.requires_grad = True

In [ ]:
model = MyModel(num_classes=1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
path = '/kaggle/input/coatnet-head-age/coatnet_head_age.pth'
model.load_state_dict(torch.load(path, map_location=device))
model = model.to(device)


In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(
    [
        {'params': model.backbone.parameters(), 'lr': 0.000003},
        {'params': model.classifier.parameters(), 'lr':0.00001}
    ], weight_decay=0.1,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, min_lr = 1e-8, verbose=True
)
train_loss, val_loss = train(model, train_loader, val_loader, criterion, optimizer, epochs=30, device=device)

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(train_loss) + 1)
plt.figure(figsize=(10,5))
plt.plot(epochs, train_loss, label='Train loss')
plt.plot(epochs, val_loss, label='Val loss')
plt.xlabel('Epochs')
plt.ylabel('Training vs Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
criterion = nn.MSELoss()
criterion2 = nn.L1Loss()
test_loss = 0.0
test_loss2 = 0.0
correct = 0
total = 0
model.eval()

with torch.no_grad():
    for images, labels in test_loader:  
        images, labels = images.to(device), labels.to(device).float().unsqueeze(1)
        outputs = model(images)
        loss = criterion(outputs, labels)  
        loss2 = criterion2(outputs, labels)
        test_loss += loss.item() * images.size(0)
        test_loss2 += loss2.item() * images.size(0)
        total += labels.size(0)

avg_test_loss = test_loss / total
avg_test_loss2 = test_loss2 / total
print(f"Test loss: {avg_test_loss:.4f}")
print(f"Test loss2: {avg_test_loss2:.4f}")

# Results

### Full models MAE:
**ConvNeXt - 4.95**

**Swin     - 5.64**

**CoAtNet  - 5.37**


### Full models MSE:

**ConvNeXt - 51.13**

**Swin     - 64.12**

**CoAtNet  - 57.49**